# TFM - Feature Engineering

## Objective

The main objective of this notebook is to prepare the final integrated MLB hitters dataset for predictive modeling by defining the target variable, selecting relevant predictors and applying only the necessary transformations.

This phase does not aim to train Machine Learning models yet, but to create a consistent, validated and modeling-ready dataset while avoiding redundant variables and data leakage.

Notebook: 05_Feature_Engineering

Author: Ronald Báez

In [1]:
# Import libraries
import pandas as pd
import numpy as np

# Load the integrated dataset
df = pd.read_csv("../01_data/03_final_data/mlb_hitters_integrated.csv")

print(f"Dataset dimensions: {df.shape}")
print(f"Number of columns: {df.shape[1]}")

display(df.head())


Dataset dimensions: (3684, 37)
Number of columns: 37


,rk,player,age,team,lg,war,g,pa,ab,r,...,gidp,hbp,sh,sf,ibb,pos,awards,year,rownum,salary
0,384.0,a.j. ellis,36.0,MIA,NL,0.6,51.0,163.0,143.0,17.0,...,6.0,6.0,2.0,0.0,0.0,2H,no_award,2017,1,2500000
1,367.0,a.j. ellis,37.0,SDP,NL,0.2,66.0,183.0,151.0,19.0,...,2.0,1.0,3.0,2.0,1.0,2H/7D,no_award,2018,1,1250000
2,325.0,aaron altherr,25.0,PHI,NL,-0.2,57.0,227.0,198.0,23.0,...,4.0,6.0,0.0,0.0,2.0,978/H,no_award,2016,1,515500
3,207.0,aaron altherr,26.0,PHI,NL,1.9,107.0,412.0,372.0,58.0,...,12.0,7.0,0.0,1.0,2.0,798/H,no_award,2017,1,538500
4,287.0,aaron altherr,27.0,PHI,NL,-0.9,105.0,285.0,243.0,28.0,...,13.0,4.0,0.0,2.0,0.0,9H8/7,no_award,2018,1,440336


In [2]:
# Display available columns
print("Available columns:")
print(df.columns.tolist())

Available columns:
['rk', 'player', 'age', 'team', 'lg', 'war', 'g', 'pa', 'ab', 'r', 'h', '2b', '3b', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'ba', 'obp', 'slg', 'ops', 'ops_plus', 'roba', 'rbat_plus', 'tb', 'gidp', 'hbp', 'sh', 'sf', 'ibb', 'pos', 'awards', 'year', 'rownum', 'salary']


### Define Target Variable

The target variable for the predictive modeling phase is `salary`, which represents the annual salary associated with each player-season record.

The exploratory analysis showed that salary has a strongly right-skewed distribution due to the presence of a relatively small number of highly paid players. Therefore, a logarithmic transformation is created to reduce this asymmetry and limit the influence of extreme values during model training.


In [3]:
# Validate target variable
print("Missing salary values:", df["salary"].isna().sum())
print("Non-positive salary values:", (df["salary"] <= 0).sum())

df["salary"].describe()

Missing salary values: 0
Non-positive salary values: 0


count    3.684000e+03
mean     4.687144e+06
std      6.702141e+06
min      3.223000e+04
25%      5.450000e+05
50%      1.430138e+06
75%      6.000000e+06
max      4.417500e+07
Name: salary, dtype: float64

In [4]:
# Create logarithmic target variable
df["log_salary"] = np.log1p(df["salary"])

# Compare original and transformed target distributions
target_comparison = pd.DataFrame({
    "Variable": ["salary", "log_salary"],
    "Skewness": [
        df["salary"].skew(),
        df["log_salary"].skew()
    ],
    "Minimum": [
        df["salary"].min(),
        df["log_salary"].min()
    ],
    "Maximum": [
        df["salary"].max(),
        df["log_salary"].max()
    ]
})

target_comparison

,Variable,Skewness,Minimum,Maximum
0,salary,2.122460,32230.000000,4.417500e+07
1,log_salary,0.231377,10.380684,1.760367e+01


The logarithmic transformation reduces the positive skewness of the original salary distribution and produces a more balanced target variable.

For this reason, `log_salary` will be used as the main target during model training. The original `salary` variable will be retained to interpret and report predictions in monetary units.


In [5]:
# Define target variable
target = "log_salary"

### Create Derived Features

The integrated dataset already contains relevant traditional and advanced performance indicators, including batting average, on-base percentage, slugging, OPS, OPS+, and WAR. Therefore, these metrics are retained without recalculating them.

A limited number of additional features are created to represent power, plate discipline, and the potential non-linear relationship between player age and salary. All rate variables are calculated using plate appearances to control for differences in playing time.

A career experience variable is not created because the dataset does not include each player's actual MLB debut year. Using the first observed season within the available period could produce a misleading measure of experience.


In [6]:
# Create derived features
df["age_squared"] = df["age"] ** 2

df["hr_rate"] = np.where(
    df["pa"] > 0,
    df["hr"] / df["pa"],
    np.nan
)

df["bb_rate"] = np.where(
    df["pa"] > 0,
    df["bb"] / df["pa"],
    np.nan
)

df["so_rate"] = np.where(
    df["pa"] > 0,
    df["so"] / df["pa"],
    np.nan
)

In [7]:
# Review derived features
derived_features = [
    "age_squared",
    "hr_rate",
    "bb_rate",
    "so_rate"
]

df[derived_features].describe()

,age_squared,hr_rate,bb_rate,so_rate
count,3684.000000,3684.000000,3684.000000,3684.000000
mean,810.617264,0.031299,0.084010,0.222410
std,219.193906,0.016618,0.031261,0.063637
min,361.000000,0.000000,0.008065,0.039216
25%,625.000000,0.019074,0.061350,0.177057
50%,784.000000,0.030246,0.081967,0.219697
75%,961.000000,0.041667,0.103241,0.263774
max,1849.000000,0.111111,0.232000,0.483444


In [8]:
# Check missing and infinite values
print("Missing values:")
print(df[derived_features].isna().sum())

print("\nInfinite values:")
print(np.isinf(df[derived_features]).sum())

Missing values:
age_squared    0
hr_rate        0
bb_rate        0
so_rate        0
dtype: int64

Infinite values:
age_squared    0
hr_rate        0
bb_rate        0
so_rate        0
dtype: int64


### Remove Non-Predictive Variables

Columns that represent player identifiers, source-specific indexes, rankings, or potentially post-season information are removed because they do not provide stable predictive information.

The original salary variable is retained for interpretation purposes, although it will not be included among the predictors because `log_salary` has been defined as the modeling target.


In [9]:
# Remove non-predictive and potentially problematic variables
columns_to_remove = [
    "player",
    "rk",
    "rownum",
    "awards"
]

df_model = df.drop(columns=columns_to_remove).copy()

print("Original dataset shape:", df.shape)
print("Modeling dataset shape:", df_model.shape)
print("Removed columns:", columns_to_remove)

Original dataset shape: (3684, 42)
Modeling dataset shape: (3684, 38)
Removed columns: ['player', 'rk', 'rownum', 'awards']


In [10]:
# Confirm remaining columns
print("Remaining columns:")
print(df_model.columns.tolist())

Remaining columns:
['age', 'team', 'lg', 'war', 'g', 'pa', 'ab', 'r', 'h', '2b', '3b', 'hr', 'rbi', 'sb', 'cs', 'bb', 'so', 'ba', 'obp', 'slg', 'ops', 'ops_plus', 'roba', 'rbat_plus', 'tb', 'gidp', 'hbp', 'sh', 'sf', 'ibb', 'pos', 'year', 'salary', 'log_salary', 'age_squared', 'hr_rate', 'bb_rate', 'so_rate']


### Select Predictive Features

The predictor set is selected using the results of the exploratory analysis, baseball domain relevance, data quality, and the need to limit redundant information.

The final selection combines player context, playing time, overall value, offensive production, and plate discipline. Highly related metrics such as batting average, OBP, slugging, OPS, wOBA, and adjusted batting indicators are not included simultaneously because they represent overlapping aspects of offensive performance.

Team, league, and position variables are excluded to keep the model focused on individual performance and to avoid introducing unnecessary categorical complexity.


In [11]:
# Define selected predictive features
selected_features = [
    "year",
    "age",
    "age_squared",
    "pa",
    "war",
    "ops_plus",
    "rbi",
    "sb",
    "hr_rate",
    "bb_rate",
    "so_rate"
]

print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features)

Number of selected features: 11
Selected features: ['year', 'age', 'age_squared', 'pa', 'war', 'ops_plus', 'rbi', 'sb', 'hr_rate', 'bb_rate', 'so_rate']


In [12]:
# Keep selected features and target variables
df_model = df_model[
    selected_features + ["salary", target]
].copy()

print("Selected modeling dataset shape:", df_model.shape)

df_model.head()

Selected modeling dataset shape: (3684, 13)


,year,age,age_squared,pa,war,ops_plus,rbi,sb,hr_rate,bb_rate,so_rate,salary,log_salary
0,2017,36.0,1296.0,163.0,0.6,82.0,14.0,0.0,0.036810,0.073620,0.177914,2500000,14.731802
1,2018,37.0,1369.0,183.0,0.2,104.0,15.0,0.0,0.005464,0.142077,0.202186,1250000,14.038655
2,2016,25.0,625.0,227.0,-0.2,59.0,22.0,7.0,0.017621,0.101322,0.303965,515500,13.152895
3,2017,26.0,676.0,412.0,1.9,122.0,65.0,5.0,0.046117,0.077670,0.252427,538500,13.196545
4,2018,27.0,729.0,285.0,-0.9,69.0,38.0,3.0,0.028070,0.126316,0.319298,440336,12.995296


In [13]:
# Review selected variables
feature_summary = pd.DataFrame({
    "Feature": selected_features,
    "Data type": [df_model[column].dtype for column in selected_features],
    "Missing values": [
        df_model[column].isna().sum()
        for column in selected_features
    ]
})

feature_summary

,Feature,Data type,Missing values
0,year,int64,0
1,age,float64,0
2,age_squared,float64,0
3,pa,float64,0
4,war,float64,0
5,ops_plus,float64,0
6,rbi,float64,0
7,sb,float64,0
8,hr_rate,float64,0
9,bb_rate,float64,0


### Classify Features

All selected predictors are numerical variables. Therefore, no categorical encoding is required for the final modeling dataset.

Numerical scaling will not be applied during this phase. It will be incorporated into the Machine Learning pipelines after splitting the data into training and test sets, ensuring that preprocessing parameters are learned exclusively from the training data.

Scaling will be relevant for linear and regularized models, while tree-based models will use the original numerical values.


In [14]:
# Classify selected features
numerical_features = selected_features.copy()
categorical_features = []

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['year', 'age', 'age_squared', 'pa', 'war', 'ops_plus', 'rbi', 'sb', 'hr_rate', 'bb_rate', 'so_rate']
Categorical features: []


In [15]:
# Validate feature data types
feature_types = df_model[numerical_features].dtypes.to_frame(
    name="Data type"
)

feature_types

,Data type
year,int64
age,float64
age_squared,float64
pa,float64
war,float64
ops_plus,float64
rbi,float64
sb,float64
hr_rate,float64
bb_rate,float64


In [16]:
# Define preprocessing groups for the modeling phase
features_to_scale = numerical_features.copy()

print("Features to scale in models that require standardization:")
print(features_to_scale)

Features to scale in models that require standardization:
['year', 'age', 'age_squared', 'pa', 'war', 'ops_plus', 'rbi', 'sb', 'hr_rate', 'bb_rate', 'so_rate']


### Validate Modeling Dataset

Before exporting the modeling dataset, the selected predictors and target variable are checked for missing values, infinite values, incorrect data types, and duplicated records.

Rows containing invalid values in the selected features or target variable are removed to ensure that the final dataset can be used directly during the Machine Learning phase.


In [17]:
# Check missing and infinite values
validation_summary = pd.DataFrame({
    "Missing values": df_model.isna().sum(),
    "Infinite values": [
        np.isinf(df_model[column]).sum()
        for column in df_model.columns
    ]
})

validation_summary

,Missing values,Infinite values
year,0,0
age,0,0
age_squared,0,0
pa,0,0
war,0,0
ops_plus,0,0
rbi,0,0
sb,0,0
hr_rate,0,0
bb_rate,0,0


In [18]:
# Replace infinite values and remove incomplete records
rows_before = df_model.shape[0]

df_model = df_model.replace([np.inf, -np.inf], np.nan)

df_model = df_model.dropna(
    subset=selected_features + [target]
).reset_index(drop=True)

rows_after = df_model.shape[0]

print("Rows before validation:", rows_before)
print("Rows after validation:", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before validation: 3684
Rows after validation: 3684
Rows removed: 0


In [19]:
# Perform final consistency checks
print("Duplicated records:", df_model.duplicated().sum())

print(
    "Non-numeric predictors:",
    df_model[selected_features]
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("Final dataset shape:", df_model.shape)

Duplicated records: 0
Non-numeric predictors: []
Final dataset shape: (3684, 13)


### Prepare and Export Modeling Dataset

The predictor matrix and target vector are created using the selected features and log_salary as the target. The final modeling dataset is exported for use in the Machine Learning phase.


In [20]:
# Create predictors and target
X = df_model[selected_features]
y = df_model[target]

# Export modeling-ready dataset
output_path = "../01_data/03_final_data/mlb_hitters_modeling.csv"
df_model.to_csv(output_path, index=False)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Dataset exported to:", output_path)

X shape: (3684, 11)
y shape: (3684,)
Dataset exported to: ../01_data/03_final_data/mlb_hitters_modeling.csv
